# Detección de Motociclistas con YOLO11

Este notebook implementa un sistema de **detección de motociclistas** usando únicamente el modelo base de detección de objetos YOLO11.

**Flujo de trabajo:**
1. Detección con modelo base YOLO11 preentrenado en COCO (personas y motocicletas)
2. Descarga del dataset de motociclistas desde Roboflow
3. Entrenamiento (fine-tuning) del modelo con el dataset personalizado
4. Evaluación del modelo entrenado
5. Inferencia con el modelo personalizado

# INSTALACIÓN DE DEPENDENCIAS

In [ ]:
!pip install ultralytics

# LIBRERÍAS

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import torch
import os

# DETECCIÓN DE OBJETOS - MODELO BASE YOLO11

El modelo YOLO11 preentrenado en COCO puede detectar **personas** (clase 0) y **motocicletas** (clase 3) sin entrenamiento adicional.  
Esto sirve como punto de partida antes del fine-tuning con el dataset personalizado de motociclistas.

In [ ]:
# Cargar el modelo base de detección de objetos YOLO11
model = YOLO("yolo11n.pt")

# Clases COCO relevantes para motociclistas
# 0 = person,  3 = motorcycle
CLASES = [0, 3]

# Imagen de prueba: reemplaza con una imagen que contenga motociclistas
IMAGEN = "https://ultralytics.com/images/bus.jpg"

results = model.predict(
    IMAGEN,
    classes=CLASES,
    conf=0.4
)

for result in results:
    boxes  = result.boxes
    xywh   = boxes.xywh   # centro-x, centro-y, ancho, alto
    xywhn  = boxes.xywhn  # normalizado
    xyxy   = boxes.xyxy   # esquina sup-izq y esquina inf-der
    xyxyn  = boxes.xyxyn  # normalizado
    names  = [result.names[cls.item()] for cls in boxes.cls.int()]
    confs  = boxes.conf

    print(f"Objetos detectados : {names}")
    print(f"Confianzas         : {[round(c.item(), 2) for c in confs]}")

    img_annotated = result.plot()

    plt.figure(figsize=(12, 8))
    plt.imshow(img_annotated[..., ::-1])  # BGR a RGB
    plt.axis("off")
    plt.title("DETECCION DE OBJETOS - Modelo Base YOLO11 (person + motorcycle)")
    plt.show()

# SUBIR Y DESCOMPRIMIR EL DATASET EN COLAB

Como el dataset fue descargado desde un proyecto público de Roboflow Universe (no es tuyo), no se puede usar la API. En su lugar, se sube el archivo `.zip` directamente a Colab.

**Archivo esperado:** `Final Motorcyclist Model.v1i.yolov11.zip`

**Pasos:**
1. Al ejecutar la celda siguiente, se abrirá un selector de archivos
2. Selecciona el `.zip` desde tu computadora
3. Espera a que termine la subida (puede tardar unos minutos)
4. El script descomprime el zip y localiza el `data.yaml` automáticamente

> **Nota:** si el zip es muy grande (>200 MB), conviene subirlo primero a Google Drive y montar Drive en Colab.

In [ ]:
from google.colab import files
import zipfile, os

# === 1. Subir el zip del dataset ===
# El zip esperado se llama: "Final Motorcyclist Model.v1i.yolov11.zip"
print("Selecciona el archivo .zip del dataset desde tu computadora:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f"\nArchivo subido: {zip_name}")

# === 2. Descomprimir ===
EXTRACT_TO = "/content/dataset"
os.makedirs(EXTRACT_TO, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_TO)

# === 3. Localizar el data.yaml automaticamente ===
DATASET_PATH = None
for root, dirs, file_list in os.walk(EXTRACT_TO):
    if "data.yaml" in file_list:
        DATASET_PATH = root
        break

if DATASET_PATH:
    print(f"\nDataset listo en : {DATASET_PATH}")
    print(f"data.yaml        : {DATASET_PATH}/data.yaml")
else:
    print("\nERROR: no se encontro data.yaml dentro del zip")

## Estructura del Dataset

El dataset descargado debe tener la siguiente estructura de carpetas y un archivo `data.yaml` de configuración.

In [ ]:
print("Estructura del dataset:")
print("dataset/")
print("  data.yaml          - configuracion del dataset (clases y rutas)")
print("  train/")
print("    images/          - imagenes de entrenamiento")
print("    labels/          - anotaciones .txt en formato YOLO")
print("  valid/")
print("    images/          - imagenes de validacion")
print("    labels/")
print("  test/")
print("    images/          - imagenes de prueba")
print("    labels/")
print()

yaml_path = os.path.join(DATASET_PATH, "data.yaml")
if os.path.exists(yaml_path):
    print("Contenido de data.yaml:")
    with open(yaml_path, "r") as f:
        print(f.read())

# ENTRENAMIENTO DEL MODELO

Se realiza **transfer learning** partiendo del modelo YOLO11n preentrenado en COCO.  
El modelo se ajusta (fine-tuning) con el dataset de motociclistas para mejorar la detección específica.

| Parámetro   | Descripción                                              |
|-------------|----------------------------------------------------------|
| `epochs`    | Número de pasadas completas sobre el dataset             |
| `imgsz`     | Tamaño al que se redimensionan las imágenes              |
| `batch`     | Imágenes procesadas por iteración                        |
| `patience`  | Épocas sin mejora antes de detener el entrenamiento      |
| `device`    | GPU (0) o CPU                                            |

In [ ]:
# Cargar el modelo base para transfer learning
model = YOLO("yolo11n.pt")

# Detectar dispositivo disponible
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de entrenamiento: {'GPU (CUDA)' if device == 0 else 'CPU'}")

# Entrenar el modelo con el dataset de motociclistas
results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16 if device == 0 else 4,
    name="motociclistas_yolo11",
    patience=10,
    device=device
)

print("\nEntrenamiento completado!")
print("Mejor modelo guardado en: runs/detect/motociclistas_yolo11/weights/best.pt")

# EVALUACIÓN DEL MODELO

Se evalúan las métricas de rendimiento del modelo sobre el conjunto de validación.

- **mAP@50**: precisión media con IoU >= 0.50
- **mAP@50-95**: precisión media promediada en umbrales IoU 0.50 a 0.95
- **Precisión**: fracción de detecciones correctas sobre el total de detecciones
- **Recall**: fracción de objetos reales detectados correctamente

In [ ]:
metrics = model.val()

print("=" * 35)
print("  METRICAS DE EVALUACION")
print("=" * 35)
print(f"  mAP@50    : {metrics.box.map50:.4f}")
print(f"  mAP@50-95 : {metrics.box.map:.4f}")
print(f"  Precision : {metrics.box.mp:.4f}")
print(f"  Recall    : {metrics.box.mr:.4f}")
print("=" * 35)

# DETECCIÓN CON MODELO ENTRENADO

Se usa el modelo fine-tuned (`best.pt`) para detectar motociclistas en nuevas imágenes.

In [ ]:
# Cargar el mejor checkpoint del entrenamiento
modelo_entrenado = YOLO("runs/detect/motociclistas_yolo11/weights/best.pt")

# IMPORTANTE: Reemplaza con la ruta a tu imagen de prueba con motociclistas
IMAGEN_PRUEBA = "RUTA_A_TU_IMAGEN.jpg"

results = modelo_entrenado.predict(
    IMAGEN_PRUEBA,
    conf=0.5,
    save=True
)

for result in results:
    boxes  = result.boxes
    xywh   = boxes.xywh   # centro-x, centro-y, ancho, alto
    xywhn  = boxes.xywhn  # normalizado
    xyxy   = boxes.xyxy   # esquina sup-izq y esquina inf-der
    xyxyn  = boxes.xyxyn  # normalizado
    names  = [result.names[cls.item()] for cls in boxes.cls.int()]
    confs  = boxes.conf

    print(f"Motociclistas detectados : {len(boxes)}")
    print(f"Clases                   : {names}")
    print(f"Confianzas               : {[round(c.item(), 2) for c in confs]}")

    img_annotated = result.plot()

    plt.figure(figsize=(12, 8))
    plt.imshow(img_annotated[..., ::-1])  # BGR a RGB
    plt.axis("off")
    plt.title("DETECCION DE MOTOCICLISTAS - Modelo Fine-tuned")
    plt.show()

# PREGUNTAS

1. ¿Qué diferencia hay entre la detección del modelo base (COCO) y el modelo entrenado con el dataset personalizado?
2. ¿En qué aplicaciones reales podría utilizarse un sistema de detección de motociclistas?
3. ¿Qué ventajas tiene el transfer learning sobre entrenar un modelo desde cero?